# 4 — UMAP panels (per-query and combined)

Renders per-query UMAPs on `X_umap_scvi` (the integrated latent UMAP from notebook 2) coloured by `scanvi_prediction`, plus a combined panel using a thin obs-only DataFrame from `combine_predictions_obs` so we never load full counts.

Output: `figures/d2-d4-integrated/umap-panels/*.pdf`

In [ ]:
import sys, gc
from pathlib import Path

_p = Path('.').resolve()
while not (_p / 'src' / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import CELLASSIGN_DIR, FIGURES_DIR, INTEGRATION_DIR
from src.palette import celltype_palette, normalize_celltype_name
from src.transfer_learning import combine_predictions_obs

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

QUERY_KEYS = ['D2_DZ', 'D2_Lapa', 'D4_DZ', 'D4_Lapa']
REFERENCE_H5AD = INTEGRATION_DIR / 'scarches_model' / 'reference_with_latent.h5ad'

fig_dir = FIGURES_DIR / 'd2-d4-integrated' / 'umap-panels'
fig_dir.mkdir(parents=True, exist_ok=True)

QUERY_PATHS = [CELLASSIGN_DIR / f'{k.lower()}_scanvi_predictions.h5ad' for k in QUERY_KEYS]

In [ ]:
# Per-query UMAP coloured by scanvi_prediction
for key, path in zip(QUERY_KEYS, QUERY_PATHS):
    a = sc.read_h5ad(path)
    # Use the integrated-latent UMAP as the main view
    if 'X_umap_scvi' in a.obsm:
        a.obsm['X_umap'] = a.obsm['X_umap_scvi']
    sc.settings.figdir = str(fig_dir)
    sc.pl.umap(
        a,
        color='scanvi_prediction',
        palette=celltype_palette,
        title=f'{key}: scANVI predictions on integrated UMAP',
        save=f'_{key.lower()}_scanvi.pdf',
        show=True,
    )
    sc.pl.umap(
        a,
        color='scanvi_confidence',
        cmap='viridis',
        title=f'{key}: scANVI confidence',
        save=f'_{key.lower()}_confidence.pdf',
        show=True,
    )
    del a; gc.collect()

In [ ]:
# Build a thin combined obs DataFrame (counts NOT loaded). Includes the reference for context.
df = combine_predictions_obs(
    query_h5ad_paths=QUERY_PATHS,
    reference_h5ad_path=REFERENCE_H5AD,
)
df['Time_point'] = df['Time_point'].astype(str)
df['Treatment']  = df['Treatment'].astype(str)
df['scanvi_prediction'] = df.get('scanvi_prediction', df.get('scanvi_prediction_ref'))
df.shape, df['dataset'].value_counts()

In [ ]:
# Combined per-dataset UMAP grid (faceted by dataset). Coords come from each h5ad's X_umap_scvi.
umap_x = next((c for c in df.columns if c.endswith('_0')), None)
umap_y = next((c for c in df.columns if c.endswith('_1')), None)
assert umap_x and umap_y, 'no UMAP coordinates were found in the combined obs DataFrame'

datasets_in_plot = ['D10_Lapa', 'D2_DZ', 'D2_Lapa', 'D4_DZ', 'D4_Lapa']
fig, axes = plt.subplots(1, len(datasets_in_plot), figsize=(4*len(datasets_in_plot), 4), sharey=False)
for ax, key in zip(axes, datasets_in_plot):
    sub = df[df['dataset'] == key]
    if sub.empty:
        ax.set_visible(False); continue
    label_col = 'scanvi_prediction' if 'scanvi_prediction' in sub.columns else 'initial_cellassign_prediction'
    labels = sub[label_col].astype(str).map(normalize_celltype_name)
    colors = labels.map(lambda n: celltype_palette.get(n, '#808080'))
    ax.scatter(sub[umap_x], sub[umap_y], c=colors, s=2, linewidths=0)
    ax.set_title(key); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('per-dataset UMAPs on integrated latent space')
plt.tight_layout()
plt.savefig(fig_dir / 'combined_per_dataset_umap.pdf')
plt.show()

In [ ]:
# Highlight panels for rare classes (Goblet cells, EECs) per query
rare_classes = ['Goblet cells', 'EECs', 'Enterocytes']
for key, path in zip(QUERY_KEYS, QUERY_PATHS):
    a = sc.read_h5ad(path)
    if 'X_umap_scvi' in a.obsm:
        a.obsm['X_umap'] = a.obsm['X_umap_scvi']
    fig, axes = plt.subplots(1, len(rare_classes), figsize=(4*len(rare_classes), 4))
    coords = a.obsm['X_umap']
    raw = a.obs['scanvi_prediction_raw'].astype(str)
    for ax, cls in zip(axes, rare_classes):
        mask = raw == cls
        ax.scatter(coords[~mask, 0], coords[~mask, 1], s=2, c='lightgray', linewidths=0)
        ax.scatter(coords[mask, 0], coords[mask, 1], s=4, c=celltype_palette.get(cls, '#000'), linewidths=0)
        ax.set_title(f'{cls} (n={int(mask.sum())})'); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'{key}: rare-class highlights')
    plt.tight_layout()
    plt.savefig(fig_dir / f'{key.lower()}_rare_highlights.pdf')
    plt.show()
    del a; gc.collect()